# Users Feature engineering

We will create the columns that are intersing for our users

In [100]:
import pandas as pd

In [101]:
sessions = pd.read_csv('data/sessions.csv')
users = pd.read_csv('data/users.csv')
flights = pd.read_csv('data/flights.csv')
hotels = pd.read_csv('data/hotels.csv')

id from trips that were cancelled

In [102]:
cancelled_trip_ids = sessions[sessions['cancellation']]['trip_id']

## Columns directly from users table

In [103]:
users.columns

Index(['user_id', 'birthdate', 'gender', 'married', 'has_children',
       'home_country', 'home_city', 'home_airport', 'home_airport_lat',
       'home_airport_lon', 'sign_up_date'],
      dtype='str')

directly took columns from user table

In [104]:
user_features = users[['user_id', 'gender', 'married', 'has_children']].copy()

## Tenure

In [105]:
user_features['tenure'] = (pd.Timestamp.today() - pd.to_datetime(users['sign_up_date'])).dt.days # feature column

Idea: From this column create group of new customers.

## Overcarrier

In [106]:
flights

,trip_id,origin_airport,destination,destination_airport,seats,return_flight_booked,departure_time,return_time,checked_bags,trip_airline,destination_airport_lat,destination_airport_lon,base_fare_usd
0,363535-ae2567b185da4e3994607ce71f98a96b,CLT,phoenix,PHX,1,False,2023-01-13 15:00:00.000000,NaN,0,Alaska Airlines,33.535,-112.383,251.87
1,549482-f5e7931dd7b6460b90b89ea0aaabfc78,YVR,san diego,SAN,1,True,2023-07-25 11:00:00.000000,2023-07-31 11:00:00.000000,1,American Airlines,32.699,-117.215,369.26
2,585745-7f13ca3cb64441e08ba34a3020e7ff7b,LBB,new york,JFK,1,True,2023-07-24 07:00:00.000000,2023-07-29 07:00:00.000000,0,Allegiant Air,40.640,-73.779,407.51
3,596153-eb736cf7fe6f4aaf8c1638e399111ced,YKZ,calgary,YYC,1,True,2023-07-25 15:00:00.000000,2023-07-27 15:00:00.000000,1,WestJet,51.114,-114.020,449.58
4,636526-80cabddfe58c406d9ba90c77f85c66ff,LGA,los angeles,LAX,1,True,2023-07-23 13:00:00.000000,2023-07-26 13:00:00.000000,1,Air New Zealand,33.942,-118.408,691.85
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13188,544449-e96a99d225be4ca2862bfa16d02f94d9,LGA,chicago,MDW,2,True,2023-03-30 08:00:00.000000,2023-04-01 08:00:00.000000,0,AirTran Airways,41.786,-87.752,389.69
13189,566735-978b549aa0804742883e1a10e57d6790,BOS,indianapolis,IND,1,True,2023-04-02 09:00:00.000000,2023-04-06 09:00:00.000000,1,Allegiant Air,39.717,-86.294,175.15
13190,580611-b8fc699133a8430a935067cd85f68549,JFK,montreal,YUL,1,True,2023-03-31 11:00:00.000000,2023-04-05 11:00:00.000000,0,Royal Jordanian,45.517,-73.417,97.40
13191,588386-233db0f627cf4aaaaeaa9a632daa5ec4,CLE,philadelphia,PHL,1,True,2023-04-04 08:00:00.000000,2023-04-06 08:00:00.000000,2,American Airlines,39.872,-75.241,106.00


In [107]:
flights_w_users = pd.merge(flights, sessions[['trip_id','user_id']], on='trip_id', how='left')

filter out valid trips and calculate average bags per seat

In [108]:
seats_avg = flights_w_users[~flights_w_users['trip_id'].isin(cancelled_trip_ids)].groupby('user_id')['seats'].mean()

In [109]:
checked_bags_avg = flights_w_users[~flights_w_users['trip_id'].isin(cancelled_trip_ids)].groupby('user_id')['checked_bags'].mean()

In [12]:
bags_per_seat = checked_bags_avg/seats_avg

In [110]:
overcarrier = bags_per_seat > 2  # feature column

overcarrier.name = "overcarrier"

## Frequent travelers

In [111]:
total_trips  = sessions.groupby('user_id')['trip_id'].nunique() # feature column
total_trips.name = 'total_trips'

In [112]:
number_of_cancellations = sessions.groupby('user_id')['cancellation'].sum()

In [113]:
cancellation_rate  = number_of_cancellations/total_trips  # feature column
cancellation_rate.name = 'cancellation_rate'

## Frequent destination

In [115]:
unique_flight_destination = flights_w_users.groupby('user_id')['destination'].nunique()

In [116]:
taken_flights = total_trips - number_of_cancellations

In [117]:
frequent_destinations = unique_flight_destination/taken_flights  # feature column
frequent_destinations.name = 'frequent_destinations'

## Booking rate

In [59]:
total_sessions = sessions[~sessions['trip_id'].isin(cancelled_trip_ids)].groupby("user_id")['session_id'].count()

In [60]:
nan_sessions = sessions[~sessions['trip_id'].isin(cancelled_trip_ids)].groupby('user_id')['trip_id'].apply(lambda x: x.isna().sum())

In [118]:
sessions_booking_rate = 1 - nan_sessions/total_sessions # feature column
sessions_booking_rate.name = 'sessions_booking_rate'

## Percentage discounted trip

In [119]:
sessions

,session_id,user_id,trip_id,session_start,session_end,flight_discount,hotel_discount,flight_discount_amount,hotel_discount_amount,flight_booked,hotel_booked,page_clicks,cancellation
0,519853-4c326bf41c934dbc8be18e225304e145,519853,NaN,2023-02-23 21:17:00.000000,2023-02-23 21:18:00.000000,False,True,NaN,0.10,False,False,8,False
1,519961-d51f73bd907c4d1b870482986df44934,519961,NaN,2023-02-23 10:33:00.000000,2023-02-23 10:37:26.000000,False,False,NaN,NaN,False,False,36,False
2,520819-9370019847834216ae16820953e4003d,520819,NaN,2023-02-23 18:55:00.000000,2023-02-23 18:56:22.000000,True,False,0.1,NaN,False,False,11,False
3,521768-a256b0bd9f7a4b79a1ef41dc75cb4cde,521768,NaN,2023-02-23 21:31:00.000000,2023-02-23 21:31:37.000000,False,False,NaN,NaN,False,False,5,False
4,522438-517e5c924fc14b44a97d2729b3712ae7,522438,522438-818ce6433b8849e0a5ec093b4c3ac782,2023-02-23 10:04:00.000000,2023-02-23 10:06:19.000000,False,True,NaN,0.15,True,True,19,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47431,516579-c23ab895874c4e7da3d0ce50a693d750,516579,NaN,2023-02-23 20:00:00.000000,2023-02-23 20:01:15.000000,False,False,NaN,NaN,False,False,10,False
47432,518127-66b3a35543284123bfb8e615927c017d,518127,NaN,2023-02-23 12:31:00.000000,2023-02-23 12:31:31.000000,True,True,0.1,0.05,False,False,4,False
47433,518907-c80deed1d66c42cfa469693143c03449,518907,NaN,2023-02-23 09:54:00.000000,2023-02-23 09:54:25.000000,False,False,NaN,NaN,False,False,3,False
47434,519137-ffd5eb50b1894137810d6a0a10c2f35f,519137,519137-f9aa897856104fa39d7970dd058a1fea,2023-02-23 15:18:00.000000,2023-02-23 15:20:12.000000,False,True,NaN,0.05,True,True,18,False


In [120]:
valid_bookings = sessions[(~sessions['trip_id'].isin(cancelled_trip_ids)) & (~sessions['trip_id'].isna())].copy()

In [121]:
valid_bookings['discount']  = valid_bookings["flight_discount"] | valid_bookings["hotel_discount"]

In [122]:
discount_booking_rate = valid_bookings.groupby("user_id")['discount'].mean() # feature column
discount_booking_rate.name = 'discount_booking_rate'

# Create user features table by joining all the features together

In [123]:
user_features

,user_id,gender,married,has_children,tenure
0,94883,F,True,False,1501
1,101486,F,True,True,1491
2,101961,F,True,False,1491
3,106907,F,True,True,1484
4,118043,F,False,True,1470
...,...,...,...,...,...
5777,780167,F,True,True,1059
5778,785186,F,True,True,1057
5779,792549,F,False,False,1054
5780,811077,F,True,True,1048


Example of merge

In [85]:
user_features.merge(discount_booking_rate, on='user_id', how='left')

,user_id,gender,married,has_children,tenure,discount
0,94883,F,True,False,1501,0.0
1,101486,F,True,True,1491,0.0
2,101961,F,True,False,1491,0.2
3,106907,F,True,True,1484,NaN
4,118043,F,False,True,1470,0.6
...,...,...,...,...,...,...
5777,780167,F,True,True,1059,0.5
5778,785186,F,True,True,1057,0.0
5779,792549,F,False,False,1054,0.0
5780,811077,F,True,True,1048,0.0


If there is a "No name" error you can solve it with the following line

In [92]:
sessions_booking_rate.name = 'sessions_booking_rate'

In [93]:
user_features.merge(sessions_booking_rate, on='user_id', how='left')

,user_id,gender,married,has_children,tenure,sessions_booking_rate
0,94883,F,True,False,1501,0.250
1,101486,F,True,True,1491,0.250
2,101961,F,True,False,1491,0.625
3,106907,F,True,True,1484,0.000
4,118043,F,False,True,1470,0.625
...,...,...,...,...,...,...
5777,780167,F,True,True,1059,0.250
5778,785186,F,True,True,1057,0.250
5779,792549,F,False,False,1054,0.500
5780,811077,F,True,True,1048,0.125


Real join of all features we created:
- 'discount_booking_rate'
- 'overcarrier'
-  'total_trips'
- 'cancellation_rate'
- 'frequent_destinations'
- 'sessions_booking_rate'

In [127]:
user_features = user_features.merge(overcarrier, on='user_id', how='left')\
    .merge(cancellation_rate, on='user_id', how='left')\
    .merge(frequent_destinations, on='user_id', how='left')\
    .merge(sessions_booking_rate, on='user_id', how='left')\
    .merge(discount_booking_rate, on='user_id', how='left')\
    .merge(total_trips, on='user_id', how='left')

In [128]:
user_features.to_csv("data/user_features.csv", index=False)